<a href="https://colab.research.google.com/github/Darioldeveloper-dev/CSC4110-G01/blob/main/G01_U9_1_packaging_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 4110 Machine Learning
## Group G01: Student Performance Factors
### Task 9.1: Model Packaging and Web App

- **Registration number:** COM/0032/23
- **owner:** Abdul Razak (Member 4)
- **Module:** 10

**Contents**
1. Setup and the six app features
2. Final pipeline
3. Save and reload with joblib
4. Streamlit app and requirements
5. Check the app and the evidence files
6. Written answer

In [10]:
# Mount Drive, import libraries and set the project paths
from google.colab import drive
drive.mount('/content/drive')
import joblib
import sklearn
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

base = Path("/content/drive/MyDrive/CSC4110_G01")
df = pd.read_csv(base / "data" / "stage1_students.csv")
df['high_score'] = (df['Exam_Score'] >= 70).astype(int)

# The app asks for exactly six inputs, so the deployed model uses exactly those six features
numeric_cols = ['Hours_Studied', 'Attendance', 'Previous_Scores']
ordinal_cols = ['Motivation_Level', 'Parental_Involvement', 'Access_to_Resources']
features = numeric_cols + ordinal_cols
X, y = df[features], df['high_score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)
print(df[numeric_cols].agg(['min', 'max']))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(5285, 6) (1322, 6)
     Hours_Studied  Attendance  Previous_Scores
min              1          60               50
max             44         100              100


In [11]:
# Part (a) step 1: one full pipeline = preprocessing + model, fitted on the training data only
final_pipe = Pipeline([
    ('prep', ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_cols),
        ('ord', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                          ('encode', OrdinalEncoder(categories=[['Low', 'Medium', 'High']] * 3))]), ordinal_cols)])),
    ('model', LogisticRegression(max_iter=1000, random_state=42))])
final_pipe.fit(X_train, y_train)

# Test scores at the default 0.5 threshold and at the 0.30 threshold chosen in Task 5
proba = final_pipe.predict_proba(X_test)[:, 1]
rows = {}
for t in [0.5, 0.3]:
    p = (proba >= t).astype(int)
    rows[t] = {'accuracy': accuracy_score(y_test, p), 'precision': precision_score(y_test, p),
               'recall': recall_score(y_test, p), 'f1': f1_score(y_test, p)}
print(pd.DataFrame(rows).T.round(4))
print("AUC:", round(roc_auc_score(y_test, proba), 4))

     accuracy  precision  recall      f1
0.5    0.9024     0.8475  0.7354  0.7875
0.3    0.9024     0.7707  0.8585  0.8122
AUC: 0.95


In [12]:
# Part (a) step 2: save the whole pipeline to the models folder
model_path = base / "models" / "G01_model.joblib"
joblib.dump(final_pipe, model_path)
print("Saved:", model_path, "|", round(model_path.stat().st_size / 1024, 1), "KB")

Saved: /content/drive/MyDrive/CSC4110_G01/models/G01_model.joblib | 4.8 KB


In [13]:
# Part (a) step 3: reload the file and predict one test row to prove that it works
loaded = joblib.load(base / "models" / "G01_model.joblib")
one_row = X_test.iloc[[0]]
print(one_row.to_string(index=False))
print("Predicted class:", int(loaded.predict(one_row)[0]),
      "| probability of 70 or more:", round(float(loaded.predict_proba(one_row)[0, 1]), 4),
      "| actual:", int(y_test.iloc[0]))
assert loaded.predict_proba(one_row)[0, 1] == final_pipe.predict_proba(one_row)[0, 1]

 Hours_Studied  Attendance  Previous_Scores Motivation_Level Parental_Involvement Access_to_Resources
            13          68               95           Medium               Medium              Medium
Predicted class: 0 | probability of 70 or more: 0.0004 | actual: 0


In [14]:
# Part (b) steps 1-3: write the Streamlit app file into the app folder
app_code = r'''
import joblib
import pandas as pd
import streamlit as st
from pathlib import Path

# The saved pipeline sits in models/ next to the app folder
MODEL_PATH = Path(__file__).resolve().parent.parent / "models" / "G01_model.joblib"
THRESHOLD = 0.30   # chosen in Task 5: a missed deserving student is worse than one extra interview

st.set_page_config(page_title="Scholarship nomination helper", page_icon=":mortar_board:")
st.title("Scholarship nomination helper")
st.write("**CSC 4110 Machine Learning, Group G01**")
st.write("Members: Dariol Temko, Raphael Wanyama, Elisha Eddie, Abdul Razak, Emmanuel Rotich")
st.write("Dataset: Student Performance Factors (about 6,600 student records)")
st.write("Enter a student's details to see how likely they are to score 70 or more.")


@st.cache_resource
def load_model():
    return joblib.load(MODEL_PATH)


model = load_model()

hours = st.slider("Hours studied per week", 1, 44, 20)
attendance = st.slider("Attendance (%)", 60, 100, 80)
previous = st.slider("Previous exam score", 50, 100, 75)
motivation = st.selectbox("Motivation level", ["Low", "Medium", "High"], index=1)
parental = st.selectbox("Parental involvement", ["Low", "Medium", "High"], index=1)
access = st.selectbox("Access to resources", ["Low", "Medium", "High"], index=1)

if st.button("Predict"):
    row = pd.DataFrame([{
        "Hours_Studied": hours, "Attendance": attendance, "Previous_Scores": previous,
        "Motivation_Level": motivation, "Parental_Involvement": parental, "Access_to_Resources": access}])
    prob = float(model.predict_proba(row)[0, 1])
    st.metric("Probability of scoring 70 or more", f"{prob:.1%}")
    if prob >= THRESHOLD:
        st.success("Suggested: invite this student for a scholarship interview.")
    else:
        st.info("Not suggested for interview on these details alone. Teachers should still use their own judgement.")
    st.caption("This tool supports a decision, it does not make one. It must never be used to lower "
               "expectations for any student.")
'''
(base / 'app' / 'app.py').write_text(app_code.lstrip())
print('Wrote', base / 'app' / 'app.py', '-', len(app_code.splitlines()), 'lines')

Wrote /content/drive/MyDrive/CSC4110_G01/app/app.py - 44 lines


In [15]:
# Part (b) step 4: requirements.txt (scikit-learn is pinned to the version that saved the model)
requirements = f"streamlit\nscikit-learn=={sklearn.__version__}\njoblib\npandas\n"
(base / 'app' / 'requirements.txt').write_text(requirements)
print(requirements)

streamlit
scikit-learn==1.6.1
joblib
pandas



In [16]:
# Check the app headlessly: run it, click Predict and confirm there are no errors
!pip install -q streamlit
from streamlit.testing.v1 import AppTest
at = AppTest.from_file(str(base / 'app' / 'app.py'), default_timeout=60).run()
assert not at.exception, at.exception
at.button[0].click().run()
assert not at.exception, at.exception
print("App ran without errors. Result shown:", at.metric[0].value)

2026-11-23 22:46:14.996 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


App ran without errors. Result shown: 3.5%


In [17]:
# Graded evidence: list the files this task must leave behind
for f in ['models/G01_model.joblib', 'app/app.py', 'app/requirements.txt', 'outputs/app_screenshot.png']:
    print(('OK       ' if (base / f).exists() else 'MISSING  ') + f)
print("outputs/app_screenshot.png is added by hand: run the app, take a screenshot, save it in outputs.")

OK       models/G01_model.joblib
OK       app/app.py
OK       app/requirements.txt
MISSING  outputs/app_screenshot.png
outputs/app_screenshot.png is added by hand: run the app, take a screenshot, save it in outputs.


In [18]:
# Self-check: pipeline, saved file and app file
assert hasattr(final_pipe, 'predict_proba') and 'Exam_Score' not in features
assert (base / 'models' / 'G01_model.joblib').exists() and (base / 'app' / 'app.py').exists()
assert set(['streamlit', 'scikit-learn', 'joblib', 'pandas']) <= {l.split('==')[0] for l in requirements.split()}
print("Task 9.1 checks passed")

Task 9.1 checks passed


### End-to-End Workflow and Pipeline Packaging

Our group followed a structured end-to-end machine learning workflow: we first defined the problem and loaded the student dataset, then inspected, cleaned, and transformed the raw records through feature engineering and exploratory analysis. Next, we split the data without leakage, trained and compared multiple model families using cross-validation and a single final check on the held-out test set, tuned thresholds, audited fairness, and packaged the final artifact for deployment and ongoing monitoring. For deployment, we saved the entire `scikit-learn` pipeline (`models/G01_model.joblib`) rather than the classifier alone so that preprocessing steps—such as imputation, ordinal/one-hot encoding of raw text inputs like `"High"`, and feature scaling—are bundled directly with the estimator. Because the web app passes raw user inputs straight from the form, saving the model without its preprocessing pipeline would feed unencoded, unscaled values into the classifier and produce corrupted predictions. (As a practical deployment trade-off, training this app pipeline on the six user-entered inputs avoids fabricating values for 16 missing fields, lowering baseline accuracy from `0.9750` to `0.9024` at threshold `0.5` while using Task 5's `0.3` threshold to recover `0.8585` recall and `0.9500` AUC.)